# Simplified Copy-Trade Backtest

Simulates copying trades of a hand-picked set of wallets against the
processed Polygon trade shards.

## Strategy
- When a watched wallet prints a **BUY** on token `(condition_id, outcome)` at price `p`
  we place a buy order at a **slightly worse** price: `order_price = clip(p + WORSE_PRICE_OFFSET, 0.001, 0.999)`.
- The order is matched against the **first subsequent trade** whose effective price is
  `<= order_price` within `FILL_HORIZON_SECONDS`.
  - *same-token* liquidity: later BUY trades on the same `(condition_id, outcome)`.
  - *opposite-token* liquidity: later SELL trades on the **complementary** outcome
    (binary markets only), with effective price = `1 - sell_price`.
- For a wallet **SELL** the mirror applies: order at `p - WORSE_PRICE_OFFSET`, filled
  by the first later trade with effective price `>= order_price`.
  - same-token: later SELL prints on the same outcome.
  - opposite-token: later BUY prints on the complementary outcome, effective price `1 - buy_price`.

## Sharding
Trades are partitioned by the first hex character of `condition_id` (after `0x`).
All shards are processed in parallel; within each shard the backtest is exact
because a `condition_id` always falls in exactly one shard file.

## Outputs
One event ledger DataFrame with columns:
- `row_type` — `trigger` (watched-wallet trade that generated an order) or `fill` (our fill).
- `order_id` — unique integer identifying the copy order.
- `wallet` — originating wallet (`fill` rows carry the trigger wallet for reference).
- `condition_id`, `outcome`, `side`, `dt`, `tx_hash`, `price`.
- `order_price` — the price our order was placed at (`NaN` for fill rows).
- `fill_source` — `same_token` / `opposite_token` / `NaN` for trigger rows.
- `token_winner` — market resolution flag.
- `fill_pnl_usdc` — PnL of *our* position on fill rows (NaN for trigger rows),
  computed as stake × ( 1/exec_price − 1 ) for winning tokens, −stake for losers.

## Visualisation
Cumulative PnL chart with two series:
- **Wallet cohort** — raw `trade_pnl` of the watched wallets.
- **Copy strategy** — `fill_pnl_usdc` of our simulated fills.


## Configuration

In [1]:
%load_ext autoreload
%autoreload 2

import datetime
from pathlib import Path

# ── Input wallets ─────────────────────────────────────────────────────────────
# Replace with any wallet addresses you want to copy.
# Example: load from a CSV, a workspace registry, or define inline.
WATCHED_WALLETS: list[str] = None

ONLY_BUY_TRIGGERS = True
MAX_EXPOSURE_PER_WALLET_1h = 100

# ── Test period ───────────────────────────────────────────────────────────────
# None = use entire dataset (start/end are inferred from the data).
# Set to datetime.date objects to restrict the window.
PERIOD_START: datetime.date | None = datetime.date(2026, 2, 16)
# PERIOD_START: datetime.date | None = datetime.date(2026, 1, 1) # None   # e.g. datetime.date(2026, 2, 16)
PERIOD_END:   datetime.date | None = datetime.date(2026, 3, 20)
# PERIOD_END:   datetime.date | None = datetime.date(2026, 2, 16) # None   # e.g. datetime.date(2026, 3, 11)

# ── Copy-trade execution parameters ──────────────────────────────────────────
WORSE_PRICE_OFFSET: float = 0
FILL_HORIZON_SECONDS: float = 300     # max seconds after trigger to search for a fill
STAKE_USDC: float = 100.0               # max USDC per order (order qty = min(trigger_qty, STAKE_USDC / order_price))
FEE_BPS: float = 10.0                   # taker fee in basis points

# ── Data ─────────────────────────────────────────────────────────────────────
TRADES_DIR     = Path("../../data/polygon_trades_processed")
RAW_TRADES_DIR = Path("../../data/trades_polygon")

# ── Parallelism ───────────────────────────────────────────────────────────────
MAX_WORKERS: int = 10   # number of threads for parallel shard processing

In [2]:
banned_wallets = set()
if(WATCHED_WALLETS is not None):
    WATCHED_WALLETS = [w for w in WATCHED_WALLETS if w not in banned_wallets]

## Optionally load wallets from stage-1 workspace

If `WATCHED_WALLETS` is empty above, this cell loads the wallet set from the
stage-1 strategy registry. Skip or modify as needed.

In [3]:
import pandas as pd

if not WATCHED_WALLETS:
    WORKSPACE_DIR = Path("../../data/trade_signals_workspace_v2")
    wallets_path = WORKSPACE_DIR / "selected_wallets_v2.parquet"
    if wallets_path.exists():
        _wallets_df = pd.read_parquet(wallets_path, columns=["wallet"])
        WATCHED_WALLETS = _wallets_df["wallet"].tolist()
        print(f"Loaded {len(WATCHED_WALLETS)} wallets from {wallets_path.name}")
    else:
        print("No wallet source found — set WATCHED_WALLETS manually or run stage1 first.")
else:
    print(f"Using {len(WATCHED_WALLETS)} manually specified wallets.")

Loaded 218 wallets from selected_wallets_v2.parquet


## Discover shards and derive test period

In [4]:
import pandas as pd

SHARD_PATHS: list[Path] = sorted(TRADES_DIR.glob("*.parquet"))
print(f"Found {len(SHARD_PATHS)} shards under {TRADES_DIR}")

# Derive END_DATE_TRAIN from the is_train flag (last date where is_train=True).
# Test data is everything strictly after END_DATE_TRAIN.
_sample = pd.read_parquet(SHARD_PATHS[0], columns=["dt", "is_train"])
_sample["dt"] = pd.to_datetime(_sample["dt"], utc=True)
END_DATE_TRAIN: datetime.date = _sample.loc[_sample["is_train"], "dt"].max().date()
_data_end: datetime.date      = _sample["dt"].max().date()
del _sample
print(f"END_DATE_TRAIN (last train date) : {END_DATE_TRAIN}")

# Backtest always runs on test data only (strictly after END_DATE_TRAIN).
# PERIOD_START / PERIOD_END can narrow the window further, but cannot go
# earlier than the day after END_DATE_TRAIN.
#_test_start = END_DATE_TRAIN + datetime.timedelta(days=1)
period_start: datetime.date = PERIOD_START # datetime.date(2026, 2, 16) #  max(PERIOD_START, _test_start) if PERIOD_START is not None else _test_start
period_end:   datetime.date = PERIOD_END #if PERIOD_END is not None else _data_end
print(f"Backtest period : {period_start}  →  {period_end}")

Found 16 shards under ../../data/polygon_trades_processed
END_DATE_TRAIN (last train date) : 2026-02-15
Backtest period : 2026-02-16  →  2026-03-20


## Backtest engine (per-shard)

Each shard is processed independently:
1. Load only rows within the backtest period (test data only).
2. Identify trigger events (watched-wallet trades).
3. Build a per-`condition_id` opposite-outcome map (binary markets only).
4. Process triggers in time order, maintaining a copy-portfolio **position** per `(wallet, condition_id, outcome)`.
5. For each trigger, place a copy order:
   - **BUY**: `qty_ordered = min(trig_qty, STAKE_USDC / order_price)` — worst-case loss = `qty × order_price ≤ STAKE_USDC`
   - **SELL**: `qty_ordered = min(trig_qty, position, STAKE_USDC / (1 − order_price))` — worst-case loss = `qty × (1 − order_price) ≤ STAKE_USDC`. Skipped if position = 0 (no short selling).
6. Scan tape prints in time order within `FILL_HORIZON_SECONDS`. Each matching print partially fills the order: `fill_qty = min(remaining_qty, tape_qty)`.
7. One **fill** ledger row is emitted per partial fill. BUY fills increment position; SELL fills decrement it.

`fill_pnl_usdc` per fill row (all executed at `exec_price = order_price`, limit fill):
- **BUY winner**: `fill_qty × (1 − exec_price) − fill_qty × exec_price × fee`
- **BUY loser**:  `−fill_qty × exec_price × (1 + fee)`
- **SELL winner** (sold below par): `fill_qty × (exec_price − 1) − fill_qty × exec_price × fee`
- **SELL loser** (sold above par):  `fill_qty × exec_price × (1 − fee)`


In [5]:
import numpy as np
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed

# ─────────────────────────────────────────────────────────────────────────────
# Helpers
# ─────────────────────────────────────────────────────────────────────────────

def _build_complement_map(df: pd.DataFrame) -> dict[tuple[str, str], str]:
    """Return {(condition_id, outcome): opposite_outcome} for binary markets."""
    pairs: dict[str, set] = {}
    for cid, grp in df.groupby("condition_id", sort=False):
        pairs[cid] = set(grp["outcome"].dropna().unique())
    result: dict[tuple[str, str], str] = {}
    for cid, outcomes in pairs.items():
        if len(outcomes) == 2:
            a, b = sorted(outcomes)
            result[(cid, a)] = b
            result[(cid, b)] = a
    return result


def _simulate_shard(
    shard_path: Path,
    raw_tape_path: Path,
    watched_wallets: set[str],
    period_start: datetime.date,
    period_end: datetime.date,
    worse_price_offset: float,
    fill_horizon_seconds: float,
    stake_usdc: float,
    fee_bps: float,
) -> pd.DataFrame:
    """Process one shard file and return an event ledger DataFrame.

    Triggers are read from ``shard_path`` (processed per-wallet aggregated trades),
    which supplies ``token_winner`` and ``trade_pnl``.

    The fill tape is read from ``raw_tape_path`` (raw on-chain individual fills).
    Orders are simulated chronologically against the tape so that each tape print's
    quantity can only be consumed once across all active copy orders in the shard.
    """
    TRIG_COLS = [
        "wallet", "condition_id", "outcome", "dt", "side",
        "avg_price", "total_quantity", "trade_pnl", "token_winner",
        "tx_hash",
    ]
    tdf = pd.read_parquet(shard_path, columns=TRIG_COLS)
    if tdf.empty:
        return pd.DataFrame()

    tdf["dt"] = pd.to_datetime(tdf["dt"], utc=True)
    tdf = tdf.rename(columns={"avg_price": "price", "total_quantity": "quantity"})

    date_mask = (
        (tdf["dt"].dt.date >= period_start)
        & (tdf["dt"].dt.date <= period_end)
    )
    tdf = tdf[date_mask].copy()
    if tdf.empty:
        return pd.DataFrame()

    tdf["price"] = tdf["price"].astype(float)
    tdf["quantity"] = tdf["quantity"].astype(float)

    if ONLY_BUY_TRIGGERS:
        trigger_mask = (tdf["wallet"].isin(watched_wallets)) & (tdf["side"] == "BUY")
    else:
        trigger_mask = tdf["wallet"].isin(watched_wallets)
    triggers_df = tdf[trigger_mask].copy()
    if triggers_df.empty:
        return pd.DataFrame()

    tw_map: dict[tuple[str, str], bool] = {}
    for row in tdf[["condition_id", "outcome", "token_winner"]].itertuples(index=False):
        key = (row.condition_id, row.outcome)
        if key not in tw_map and row.token_winner is not None:
            tw_map[key] = bool(row.token_winner)

    complement = _build_complement_map(tdf)
    triggers_df = triggers_df.sort_values("dt", kind="mergesort").reset_index(drop=True)

    TAPE_COLS = ["condition_id", "outcome", "block_timestamp", "side", "price", "quantity", "tx_hash", "token_winner"]
    if raw_tape_path.exists():
        rdf = pd.read_parquet(raw_tape_path, columns=TAPE_COLS)
    else:
        rdf = pd.DataFrame(columns=TAPE_COLS)

    if rdf.empty:
        ledger_rows = []
        for trig in triggers_df.itertuples(index=False):
            cid = trig.condition_id
            outcome = trig.outcome
            side = trig.side
            trig_dt = trig.dt
            trig_px = float(trig.price)
            trig_qty = float(trig.quantity)
            trig_tw = bool(trig.token_winner)
            wallet = trig.wallet
            if side == "BUY":
                order_px = float(np.clip(trig_px + worse_price_offset, 0.001, 0.999))
                qty_ordered = min(trig_qty, stake_usdc / order_px)
            else:
                order_px = float(np.clip(trig_px - worse_price_offset, 0.001, 0.999))
                qty_ordered = trig_qty
            ledger_rows.append({
                "row_type": "trigger", "order_id": len(ledger_rows) + 1,
                "wallet": wallet, "condition_id": cid, "outcome": outcome,
                "side": side, "dt": trig_dt, "tx_hash": trig.tx_hash,
                "price": trig_px, "order_price": order_px,
                "qty_ordered": qty_ordered, "qty_filled": 0.0,
                "fill_qty": None, "fill_source": None,
                "token_winner": trig_tw, "exec_price": None,
                "fill_pnl_usdc": None,
                "wallet_trade_pnl": float(trig.trade_pnl) if trig.trade_pnl is not None else None,
            })
        result = pd.DataFrame(ledger_rows)
        result["shard"] = shard_path.stem
        return result

    rdf["dt"] = pd.to_datetime(rdf["block_timestamp"], unit="s", utc=True)
    rdf = rdf.drop(columns=["block_timestamp"])

    tape_start = pd.Timestamp(period_start, tz="UTC")
    tape_end = pd.Timestamp(period_end, tz="UTC") + pd.Timedelta(days=1)
    rdf = rdf[(rdf["dt"] >= tape_start) & (rdf["dt"] < tape_end)].copy()
    rdf["price"] = rdf["price"].astype(float)
    rdf["quantity"] = rdf["quantity"].astype(float)
    rdf = rdf.sort_values("dt", kind="mergesort").reset_index(drop=True)

    fee = fee_bps / 10_000.0
    horizon = pd.Timedelta(seconds=fill_horizon_seconds)
    eps = 1e-12

    ledger_rows: list[dict] = []
    order_counter = 0
    positions: dict[tuple[str, str, str], float] = defaultdict(float)

    orders: dict[int, dict] = {}
    books: dict[tuple[str, str, str], list[dict]] = defaultdict(list)

    def _append_trigger_row(order_id: int, trig, order_px: float, qty_ordered: float, trig_tw: bool) -> None:
        ledger_rows.append({
            "row_type": "trigger",
            "order_id": order_id,
            "wallet": trig.wallet,
            "condition_id": trig.condition_id,
            "outcome": trig.outcome,
            "side": trig.side,
            "dt": trig.dt,
            "tx_hash": trig.tx_hash,
            "price": float(trig.price),
            "order_price": order_px,
            "qty_ordered": qty_ordered,
            "qty_filled": None,
            "fill_qty": None,
            "fill_source": None,
            "token_winner": trig_tw,
            "exec_price": None,
            "fill_pnl_usdc": None,
            "wallet_trade_pnl": float(trig.trade_pnl) if trig.trade_pnl is not None else None,
        })

    # order can match with trade on the same side and token, or opposite side and opposite token    
    def _register_order(order_id: int, order: dict, trig_tw: bool) -> None:
        cid = order["condition_id"]
        outcome = order["outcome"]
        side = order["side"]
        books[(cid, outcome, side)].append({
            "order_id": order_id,
            "fill_source": "same_token",
            "fill_tw": trig_tw,
            "opposite": False,
        })
        opp_outcome = complement.get((cid, outcome))
        if opp_outcome is None:
            return
        opp_tw = tw_map.get((cid, opp_outcome), not trig_tw)
        opp_tape_side = "SELL" if side == "BUY" else "BUY"
        books[(cid, opp_outcome, opp_tape_side)].append({
            "order_id": order_id,
            "fill_source": "opposite_token",
            "fill_tw": trig_tw,
            "opposite": True,
        })

    def _process_tape_row(row) -> None:
        key = (row.condition_id, row.outcome, row.side)
        entries = books.get(key)
        if not entries:
            return

        tape_dt = row.dt
        tape_price = float(row.price)
        tape_remaining = float(row.quantity)
        if tape_remaining <= eps:
            return

        survivors: list[dict] = []
        for entry in entries:
            order = orders.get(entry["order_id"])
            if order is None:
                continue
            if order["remaining_qty"] <= eps:
                continue
            if order["deadline"] < tape_dt:
                continue

            eff_price = float(np.clip(1.0 - tape_price, 0.001, 0.999)) if entry["opposite"] else tape_price
            eligible = eff_price <= order["order_price"] if order["side"] == "BUY" else eff_price >= order["order_price"]

            if eligible and tape_remaining > eps:
                fill_qty = min(order["remaining_qty"], tape_remaining)
                tape_remaining -= fill_qty
                order["remaining_qty"] -= fill_qty

                if order["side"] == "BUY":
                    gross = fill_qty * (1.0 - order["order_price"]) if entry["fill_tw"] else -fill_qty * order["order_price"]
                    fill_pnl = gross - fill_qty * order["order_price"] * fee
                    positions[order["pos_key"]] += fill_qty
                else:
                    gross = fill_qty * (order["order_price"] - 1.0) if entry["fill_tw"] else fill_qty * order["order_price"]
                    fill_pnl = gross - fill_qty * order["order_price"] * fee
                    positions[order["pos_key"]] = max(positions[order["pos_key"]] - fill_qty, 0.0)

                ledger_rows.append({
                    "row_type": "fill",
                    "order_id": entry["order_id"],
                    "wallet": order["wallet"],
                    "condition_id": order["condition_id"],
                    "outcome": order["outcome"],
                    "side": order["side"],
                    "dt": tape_dt,
                    "tx_hash": row.tx_hash,
                    "price": eff_price,
                    "order_price": None,
                    "qty_ordered": order["qty_ordered"],
                    "qty_filled": None,
                    "fill_qty": fill_qty,
                    "fill_source": entry["fill_source"],
                    "token_winner": entry["fill_tw"],
                    "exec_price": order["order_price"],
                    "fill_pnl_usdc": fill_pnl,
                    "wallet_trade_pnl": None,
                })

            if order["remaining_qty"] > eps and order["deadline"] >= tape_dt:
                survivors.append(entry)

        if survivors:
            books[key] = survivors
        else:
            books.pop(key, None)

    tape_iter = iter(rdf[["condition_id", "outcome", "dt", "side", "price", "quantity", "tx_hash", "token_winner"]].itertuples(index=False))
    current_tape = next(tape_iter, None)

    for trig in triggers_df.itertuples(index=False):
        while current_tape is not None and current_tape.dt <= trig.dt:
            _process_tape_row(current_tape)
            current_tape = next(tape_iter, None)

        cid = trig.condition_id
        outcome = trig.outcome
        side = trig.side
        trig_px = float(trig.price)
        trig_qty = float(trig.quantity)
        trig_tw = bool(trig.token_winner)
        wallet = trig.wallet
        pos_key = (wallet, cid, outcome)

        if side == "BUY":
            order_px = float(np.clip(trig_px + worse_price_offset, 0.001, 0.999))
            qty_ordered = min(trig_qty, stake_usdc / order_px)
        else:
            order_px = float(np.clip(trig_px - worse_price_offset, 0.001, 0.999))
            current_pos = positions.get(pos_key, 0.0)
            if current_pos <= eps:
                continue
            sell_cap = stake_usdc / max(1.0 - order_px, 1e-9)
            qty_ordered = min(trig_qty, current_pos, sell_cap)
            if qty_ordered <= eps:
                continue

        order_counter += 1
        order_id = order_counter

        order = {
            "wallet": wallet,
            "condition_id": cid,
            "outcome": outcome,
            "side": side,
            "order_price": order_px,
            "qty_ordered": qty_ordered,
            "remaining_qty": qty_ordered,
            "deadline": trig.dt + horizon,
            "pos_key": pos_key,
        }
        orders[order_id] = order

        _append_trigger_row(order_id, trig, order_px, qty_ordered, trig_tw)
        _register_order(order_id, order, trig_tw)

    while current_tape is not None:
        _process_tape_row(current_tape)
        current_tape = next(tape_iter, None)

    if not ledger_rows:
        return pd.DataFrame()

    result = pd.DataFrame(ledger_rows)
    result["shard"] = shard_path.stem

    filled_by_order = (
        result[result["row_type"] == "fill"]
        .groupby("order_id")["fill_qty"]
        .sum()
        .rename("_total_filled")
    )
    result = result.merge(filled_by_order, on="order_id", how="left")
    trig_mask = result["row_type"] == "trigger"
    result.loc[trig_mask, "qty_filled"] = result.loc[trig_mask, "_total_filled"].fillna(0.0)
    result = result.drop(columns=["_total_filled"])
    result["qty_filled"] = result["qty_filled"].astype(float)

    return result



## Run backtest across all shards (parallel)

In [6]:
assert WATCHED_WALLETS, "WATCHED_WALLETS is empty — set wallets in the config cell or run the wallet-load cell."

watched_set = set(WATCHED_WALLETS)
shard_results: list[pd.DataFrame] = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {
        executor.submit(
            _simulate_shard,
            path,
            RAW_TRADES_DIR / path.name,
            watched_set,
            period_start,
            period_end,
            WORSE_PRICE_OFFSET,
            FILL_HORIZON_SECONDS,
            STAKE_USDC,
            FEE_BPS,
        ): path
        for path in SHARD_PATHS
    }
    for future in as_completed(futures):
        path = futures[future]
        try:
            df = future.result()
            if df is not None and not df.empty:
                shard_results.append(df)
        except Exception as exc:
            print(f"Shard {path.name} raised an exception: {exc}")
            raise

if shard_results:
    event_ledger: pd.DataFrame = (
        pd.concat(shard_results, ignore_index=True)
        .sort_values(["dt", "shard", "order_id", "row_type"])
        .reset_index(drop=True)
    )
    _key = event_ledger[["shard", "order_id"]]
    _pairs = _key.drop_duplicates().reset_index(drop=True)
    _pairs["global_order_id"] = _pairs.index + 1
    event_ledger = event_ledger.merge(_pairs, on=["shard", "order_id"], how="left")
    event_ledger["order_id"] = event_ledger["global_order_id"]
    event_ledger = event_ledger.drop(columns=["global_order_id"])
else:
    event_ledger = pd.DataFrame(columns=[
        "row_type", "order_id", "wallet", "condition_id", "outcome",
        "side", "dt", "tx_hash", "price", "order_price",
        "qty_ordered", "qty_filled", "fill_qty",
        "fill_source", "token_winner", "exec_price", "fill_pnl_usdc",
        "wallet_trade_pnl", "shard",
    ])

triggers = event_ledger[event_ledger["row_type"] == "trigger"]
fills = event_ledger[event_ledger["row_type"] == "fill"]
filled_orders = fills["order_id"].nunique()

print(f"Trigger events    : {len(triggers):>7,}")
print(f"Fill rows         : {len(fills):>7,}")
print(f"Orders with fills : {filled_orders:>7,}")
if len(triggers) > 0:
    print(f"Order fill rate   : {filled_orders/len(triggers)*100:.1f}%")
else:
    print("No trigger events found — check WATCHED_WALLETS and the period.")



Trigger events    : 151,363
Fill rows         : 141,799
Orders with fills :  75,726
Order fill rate   : 50.0%


## Summary statistics

In [7]:
fee = FEE_BPS / 10_000.0

if not fills.empty:
    filled_copy_pnl    = fills["fill_pnl_usdc"].sum()
    total_qty_filled   = fills["fill_qty"].sum()
    total_notional     = (fills["fill_qty"] * fills["exec_price"]).sum()
    orders_with_fills  = fills["order_id"].nunique()
    partial_orders     = (fills.groupby("order_id").size() > 1).sum()
    win_rate           = fills.groupby("order_id")["token_winner"].first().mean()
    avg_exec_price     = fills["exec_price"].mean()
    fill_src_counts    = fills["fill_source"].value_counts()

    # Partial-fill statistics
    trig_qty = triggers.set_index("order_id")["qty_ordered"]
    fill_qty = triggers.set_index("order_id")["qty_filled"]
    fill_pct = (fill_qty / trig_qty.clip(lower=1e-12) * 100).replace([float('inf'), float('nan')], 0)

    print(f"=== Copy-strategy performance ===")
    print(f"  Orders triggered    : {len(triggers):>7,}")
    print(f"  Orders with fills   : {orders_with_fills:>7,}  ({orders_with_fills/len(triggers)*100:.1f}%)")
    print(f"  Orders multi-fill   : {partial_orders:>7,}  ({partial_orders/max(orders_with_fills,1)*100:.1f}% of filled)")
    print(f"  Avg fill %          : {fill_pct[fill_pct>0].mean():>7.1f}%")
    print(f"  Total qty filled    : {total_qty_filled:>10,.2f} tokens")
    print(f"  Total notional      : {total_notional:>10,.2f} USDC")
    print(f"  Net PnL (USDC)      : {filled_copy_pnl:>10,.2f}")
    print(f"  Net ROI on notional : {filled_copy_pnl/total_notional*100:>8.2f}%")
    print(f"  Win rate (by order) : {win_rate*100:>8.2f}%")
    print(f"  Avg exec price      : {avg_exec_price:>8.4f}")
    print(f"\n  Fill source breakdown:")
    for src, cnt in fill_src_counts.items():
        print(f"    {src:<20s}: {cnt:>6,}  ({cnt/len(fills)*100:.1f}%)")
else:
    print("No fills — nothing to summarise.")

# Watched-wallet cohort summary
wallet_pnl = triggers["wallet_trade_pnl"].sum()
print(f"\n=== Watched-wallet cohort ===")
print(f"  Total trades        : {len(triggers):>7,}")
print(f"  Total trade PnL     : {wallet_pnl:>10,.2f} USDC")


=== Copy-strategy performance ===
  Orders triggered    : 151,363
  Orders with fills   :  75,726  (50.0%)
  Orders multi-fill   :  26,605  (35.1% of filled)
  Avg fill %          :    90.0%
  Total qty filled    : 5,240,060.27 tokens
  Total notional      : 2,242,508.47 USDC
  Net PnL (USDC)      : 279,431.92
  Net ROI on notional :    12.46%
  Win rate (by order) :    56.70%
  Avg exec price      :   0.5235

  Fill source breakdown:
    same_token          : 108,811  (76.7%)
    opposite_token      : 32,988  (23.3%)

=== Watched-wallet cohort ===
  Total trades        : 151,363
  Total trade PnL     : 2,538,249.36 USDC


## Event ledger preview

In [8]:
display_cols = [
    "row_type", "order_id", "wallet", "condition_id", "outcome", "side",
    "dt", "tx_hash", "price", "order_price", "exec_price",
    "qty_ordered", "qty_filled", "fill_qty",
    "fill_source", "token_winner", "fill_pnl_usdc", "wallet_trade_pnl",
]
available = [c for c in display_cols if c in event_ledger.columns]

# Show a few trigger+fill pairs
sample_orders = event_ledger["order_id"].unique()[:5]
event_ledger[
    event_ledger["order_id"].isin(sample_orders)
][available].sort_values(["order_id", "row_type"])


,row_type,order_id,wallet,condition_id,outcome,side,dt,tx_hash,price,order_price,exec_price,qty_ordered,qty_filled,fill_qty,fill_source,token_winner,fill_pnl_usdc,wallet_trade_pnl
2,fill,1,0xd6eef76188c7068719ac881460b7431e1ae137a2,0xf86191c4b660e0f0f4669eae5638474277a3bb31e5c3...,No,BUY,2026-02-16 00:00:51+00:00,0x0b8f56c045cc516e71896d5cf7f99a83674e137e5192...,0.960,NaN,0.961,0.004241,NaN,0.004241,same_token,True,0.000161,NaN
0,trigger,1,0xd6eef76188c7068719ac881460b7431e1ae137a2,0xf86191c4b660e0f0f4669eae5638474277a3bb31e5c3...,No,BUY,2026-02-16 00:00:15+00:00,0x0b3d6d10434abd444fe1510c39f55ce99398e88c8962...,0.961,0.961,NaN,0.004241,0.004241,NaN,None,True,NaN,0.000165
1,trigger,2,0xc7b8a3f1033a9efa9d912b60885c1d2c1e012c1f,0xbf3a613ba4ab4a066c2c3eacb877175d39a93f1fde88...,Down,BUY,2026-02-16 00:00:33+00:00,0x0ba17729dac95a428ae9e98c9c1a4004006ca120f47e...,0.410,0.410,NaN,243.902439,0.000000,NaN,None,False,NaN,-492.000000
4,fill,3,0xd6eef76188c7068719ac881460b7431e1ae137a2,0xbebb1c8c6996bdd0783db8891f4cb764afe301c7393a...,Yes,BUY,2026-02-16 00:01:41+00:00,0xf2f6a76c3c0c2a1d79d01aaf557e648c61ec3c988d28...,0.910,NaN,0.910,2.170000,NaN,2.170000,same_token,True,0.193325,NaN
3,trigger,3,0xd6eef76188c7068719ac881460b7431e1ae137a2,0xbebb1c8c6996bdd0783db8891f4cb764afe301c7393a...,Yes,BUY,2026-02-16 00:00:55+00:00,0xc0a6b59be75f38f289ce014c517b4a661ba472d8ed97...,0.910,0.910,NaN,2.170000,2.170000,NaN,None,True,NaN,0.195300
6,fill,4,0xd6eef76188c7068719ac881460b7431e1ae137a2,0xbebb1c8c6996bdd0783db8891f4cb764afe301c7393a...,Yes,BUY,2026-02-16 00:02:09+00:00,0xca8ef7824274e6e5e97d02f83f95d06d1985eacac3ba...,0.910,NaN,0.910,2.170000,NaN,2.140000,same_token,True,0.190653,NaN
5,trigger,4,0xd6eef76188c7068719ac881460b7431e1ae137a2,0xbebb1c8c6996bdd0783db8891f4cb764afe301c7393a...,Yes,BUY,2026-02-16 00:01:41+00:00,0xf2f6a76c3c0c2a1d79d01aaf557e648c61ec3c988d28...,0.910,0.910,NaN,2.170000,2.140000,NaN,None,True,NaN,0.195300
7,trigger,5,0xd6eef76188c7068719ac881460b7431e1ae137a2,0xbebb1c8c6996bdd0783db8891f4cb764afe301c7393a...,Yes,BUY,2026-02-16 00:02:09+00:00,0xca8ef7824274e6e5e97d02f83f95d06d1985eacac3ba...,0.910,0.910,NaN,2.140000,0.000000,NaN,None,True,NaN,0.192600


In [9]:
event_ledger[event_ledger['side'] == 'BUY'].groupby('wallet').agg(
    fill_pnl_usdc_sum=('fill_pnl_usdc', 'sum'),
    wallet_trade_pnl_sum=('wallet_trade_pnl', 'sum'),
    buy_count=('side', 'count')
)

,fill_pnl_usdc_sum,wallet_trade_pnl_sum,buy_count
wallet,,,
0x000d257d2dc7616feaef4ae0f14600fdf50a758e,-2093.462218,29267.033251,2074
0x023c300ddb1b8931448ff4371be11fce9467805d,-20.996505,-818.151633,71
0x027a422d069cde53d7ff6baf4ed3212d7a23ea13,447.901689,971.890027,106
0x05d7db9e8bb6e28632b798cc9cdb4aec07e5fe9f,2.907079,118.772189,13
0x06ecb7e739f5455922ce57e83284f132c7f0f845,-100.100000,-7300.000000,2
...,...,...,...
0xf41b55cf7d9fbe2fcf4895ab5f71d8644ef2b538,-378.104362,-3729.590361,250
0xf658449199d0bcf544de0ff928a3b66685f3dcfe,-4.166833,2520.029092,222
0xfb328b94ed05115259bbc48ba8182df1416edb85,841.934237,3626.350098,1251


In [40]:
event_ledger[event_ledger['side'] == 'BUY'].groupby('condition_id').agg(
    fill_pnl_usdc_sum=('fill_pnl_usdc', 'sum'),
    wallet_trade_pnl_sum=('wallet_trade_pnl', 'sum'),
    buy_count=('side', 'count')
).sort_values('fill_pnl_usdc_sum', ascending=False).head(50)

,fill_pnl_usdc_sum,wallet_trade_pnl_sum,buy_count
condition_id,,,
0x3488f31e6449f9803f99a8b5dd232c7ad883637f1c86e6953305a2ef19c77f20,187533.162538,1.089560e+06,9739
0xd4bbf7f6707c67beb736135ad32a41f6db41f8ae52d3ac4919650de9eeb94ed8,53556.810181,4.880534e+05,17615
0xc49142a0a94b68377bbc3f2e7f7f53a400274e2dcd4536cfa343be55a3c1fdf8,8014.280295,6.435801e+04,2229
0x61ce3773237a948584e422de72265f937034af418a8b703e3a860ea62e59ff36,7707.857796,2.285161e+05,6425
0x24026080b17f4e88729eab0ac2929ee37c13bfbb4a159179ec63deb4a242d9c9,4607.767835,5.436539e+04,2127
0xc5300759dc2089042380795fe7384010a6b6ebdf9e6da7ed3f786d9a5f61c563,4173.119103,7.177600e+04,3461
0x28b418b2aeb0f2eec0a64d0cca38a2a7df484a45ee5c35d3d2b2822c27c17dfb,4130.962148,8.200521e+03,1080
0xf5cd656f01d2931021935041b065d7cb17b85a7631c742f41317a95dc4d3557b,4038.231933,1.133666e+04,825
0x4b7cd1050a0c87baebe5e6008260d14b2a9d6d705cf7e7daac1be94ad5253b27,3271.568617,2.774056e+04,3192


In [10]:
event_ledger[event_ledger['side'] == 'SELL'].groupby('wallet').agg(
    fill_pnl_usdc_sum=('fill_pnl_usdc', 'sum'),
    wallet_trade_pnl_sum=('wallet_trade_pnl', 'sum'),
    sell_count=('side', 'count')
)

,fill_pnl_usdc_sum,wallet_trade_pnl_sum,sell_count
wallet,,,


In [11]:
sample_filled_orders = fills["order_id"].unique()[:50]
event_ledger[
    (event_ledger['side'] == 'SELL') &
    (event_ledger["order_id"].isin(sample_filled_orders))
].sort_values(["order_id", "dt"])

,row_type,order_id,wallet,condition_id,outcome,side,dt,tx_hash,price,order_price,qty_ordered,qty_filled,fill_qty,fill_source,token_winner,exec_price,fill_pnl_usdc,wallet_trade_pnl,shard


## Build PnL time series

In [12]:
def resample_hourly_series(df: pd.DataFrame, dt_col: str, pnl_col: str) -> pd.DataFrame:
    """Resample a PnL column to 1-h buckets and compute cumulative sum."""
    if df.empty or pnl_col not in df.columns:
        return pd.DataFrame(columns=["trade_dt", "net_pnl_usdc", "cum_net_pnl_usdc"])
    hourly = (
        df.assign(trade_dt=df[dt_col].dt.floor("1h"))
        .groupby("trade_dt", as_index=False)[pnl_col]
        .sum()
        .rename(columns={pnl_col: "net_pnl_usdc"})
        .sort_values("trade_dt")
        .reset_index(drop=True)
    )
    hourly["cum_net_pnl_usdc"] = hourly["net_pnl_usdc"].cumsum()
    return hourly


def with_zero_anchor(hourly: pd.DataFrame) -> pd.DataFrame:
    """Prepend a zero anchor one hour before the first bucket."""
    if hourly.empty:
        return hourly
    anchor = pd.DataFrame({
        "trade_dt": [hourly["trade_dt"].min() - pd.Timedelta(hours=1)],
        "net_pnl_usdc": [0.0],
        "cum_net_pnl_usdc": [0.0],
    })
    return pd.concat([anchor, hourly], ignore_index=True)


# Wallet cohort PnL: from trigger rows (their actual trade_pnl)
wallet_hourly = resample_hourly_series(triggers, "dt", "wallet_trade_pnl")

# Copy-strategy PnL: from fill rows
copy_hourly = resample_hourly_series(fills, "dt", "fill_pnl_usdc")

print(f"Wallet cohort hourly buckets : {len(wallet_hourly)}")
print(f"Copy strategy hourly buckets : {len(copy_hourly)}")

Wallet cohort hourly buckets : 792
Copy strategy hourly buckets : 792


## Cumulative PnL chart

In [13]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = go.Figure()

# ── Wallet cohort trace ───────────────────────────────────────────────────────
if not wallet_hourly.empty:
    wh = with_zero_anchor(wallet_hourly)
    fig.add_trace(go.Scatter(
        x=wh["trade_dt"],
        y=wh["cum_net_pnl_usdc"],
        mode="lines",
        line=dict(dash="dot", width=2),
        opacity=0.7,
        name="Watched wallets (raw PnL)",
        hovertemplate=(
            "<b>Watched wallets</b><br>"
            "%{x|%Y-%m-%d %H:%M}<br>"
            "cum PnL: %{y:.2f} USDC<extra></extra>"
        ),
    ))

# ── Copy-strategy trace ───────────────────────────────────────────────────────
if not copy_hourly.empty:
    ch = with_zero_anchor(copy_hourly)
    fig.add_trace(go.Scatter(
        x=ch["trade_dt"],
        y=ch["cum_net_pnl_usdc"],
        mode="lines",
        line=dict(dash="solid", width=2),
        name="Copy strategy (filled)",
        hovertemplate=(
            "<b>Copy strategy</b><br>"
            "%{x|%Y-%m-%d %H:%M}<br>"
            "cum PnL: %{y:.2f} USDC<extra></extra>"
        ),
    ))

fig.add_hline(y=0, line_dash="dash", line_color="grey", line_width=1, opacity=0.5)

fig.update_layout(
    template="plotly_white",
    height=480,
    title=dict(
        text=(
            f"Copy-trade backtest — cumulative PnL (1h) | "
            f"{period_start} → {period_end} | "
            f"{len(WATCHED_WALLETS)} wallets | "
            f"worse_price={WORSE_PRICE_OFFSET:.2f} | "
            f"horizon={int(FILL_HORIZON_SECONDS)}s"
        ),
        font=dict(size=13),
    ),
    xaxis_title="Time (1h buckets)",
    yaxis_title="Cumulative net PnL (USDC)",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
)

fig.show()

## Diagnostics

In [14]:
# Fill rate by side (order-level)
if not triggers.empty:
    trig_by_side = triggers.groupby("side").size().rename("triggers")
    fill_by_side = fills.groupby("side")["order_id"].nunique().rename("orders_filled")
    fill_rows_by_side = fills.groupby("side").size().rename("fill_rows")
    side_summary = pd.concat([trig_by_side, fill_by_side, fill_rows_by_side], axis=1).fillna(0).astype(int)
    side_summary["fill_rate"] = (side_summary["orders_filled"] / side_summary["triggers"] * 100).round(1)
    display(side_summary)


,triggers,orders_filled,fill_rows,fill_rate
side,,,,
BUY,151363,75726,141799,50.0


In [15]:
# Fill source breakdown by side
if not fills.empty:
    display(
        fills.groupby(["side", "fill_source"]).size()
        .rename("count")
        .reset_index()
        .sort_values(["side", "fill_source"])
    )

,side,fill_source,count
0,BUY,opposite_token,32988
1,BUY,same_token,108811


In [16]:
df = event_ledger.assign(
    is_trigger=event_ledger["row_type"].eq("trigger"),
    is_fill=event_ledger["row_type"].eq("fill"),
)

# --- wallet-level stats (row-based) ---
wallet_stats = (
    df.groupby("wallet")
    .agg(
        total_triggers=("is_trigger", "sum"),
        total_fills=("is_fill", "sum"),
        total_fill_pnl=("fill_pnl_usdc", "sum"),
        total_trade_pnl=("wallet_trade_pnl", "sum"),
        total_pnl=("fill_pnl_usdc", "sum"),
    )
)

# --- order-level logic (trigger -> fill) ---
order_stats = (
    df.groupby(["wallet", "order_id"])
    .agg(
        has_trigger=("is_trigger", "any"),
        has_fill=("is_fill", "any"),
    )
    .assign(trigger_with_fill=lambda x: x["has_trigger"] & x["has_fill"])
    .groupby("wallet")
    .agg(triggers_with_fill=("trigger_with_fill", "sum"))
)

# --- combine ---
result = wallet_stats.join(order_stats)

# --- derived metric ---
result["fill_ratio"] = (
    result["triggers_with_fill"] / result["total_triggers"]
).fillna(0)

result.sort_values("total_pnl", ascending=False).head()

,total_triggers,total_fills,total_fill_pnl,total_trade_pnl,total_pnl,triggers_with_fill,fill_ratio
wallet,,,,,,,
0xbacd00c9080a82ded56f504ee8810af732b0ab35,5080,7182,122489.571198,1.275992e+06,122489.571198,3236,0.637008
0x12d6cccfc7470a3f4bafc53599a4779cbf2cf2a8,4716,6858,81667.750468,4.848714e+05,81667.750468,3420,0.725191
0x7c3db723f1d4d8cb9c550095203b686cb11e5c6b,3616,7225,18811.108334,1.411745e+05,18811.108334,2258,0.624447
0xc02147dee42356b7a4edbb1c35ac4ffa95f61fa8,1416,1457,8979.187088,4.406395e+04,8979.187088,747,0.527542
0x24c8cf69a0e0a17eee21f69d29752bfa32e823e1,1173,2004,7868.128097,-4.522757e+04,7868.128097,860,0.733163


In [17]:
len(df)

293162

In [18]:
# get diff between trigger and fill timestamp per order_id
# without lambda, using groupby + agg + merge
trigger_df = df[df['row_type'] == 'trigger'].groupby('order_id')['dt'].min().reset_index()
fill_df = df[df['row_type'] == 'fill'].groupby('order_id')['dt'].min().reset_index()

merged_df = pd.merge(trigger_df, fill_df, on='order_id', how='outer', suffixes=('_trigger', '_fill'))
merged_df['diff_dt'] = merged_df['dt_fill'] - merged_df['dt_trigger']

In [19]:
merged_df[merged_df['diff_dt'].notnull()]['diff_dt'].mean().total_seconds()

66.877003

In [20]:
result[result['total_fill_pnl'] < 0].sort_values("total_fill_pnl").index.tolist()

['0x92a6294cd55b9c6ccd57efaffc365ad943118b84',
 '0x4bbe10ba5b7f6df147c0dae17b46c44a6e562cf3',
 '0xe35dfe51d8288ea409ea18a139f188a83b33f5d5',
 '0xc7b8a3f1033a9efa9d912b60885c1d2c1e012c1f',
 '0x0e3226217752d7247c67b870b72b99ba3e20535b',
 '0x3ffe1712d9b5e0d8c176f1ede6b49b9b177ae125',
 '0x000d257d2dc7616feaef4ae0f14600fdf50a758e',
 '0x80a0da00fbdc8440b0ef601341f14c3e24795708',
 '0x0a793254e040027c5d1ca0c2f87cda8e22228e05',
 '0x333e8f4fb0353ef31cf1a7d00789e1319381369b',
 '0x97d37d16d1774785197bfa23ffed625a8e493f3c',
 '0x47ab026767cc320ac6e62f6ec747d59cf4d795df',
 '0x134a63b764ac7b008356e8db1857db94e6b09e42',
 '0x2a019dc0089ea8c6edbbafc8a7cc9ba77b4b6397',
 '0x0b219cf3d297991b58361dbebdbaa91e56b8deb6',
 '0xa9964972ac7fefa772ceb380cb08cf0c0dd6e251',
 '0x41cb8653fd75ebf2a46741d224dc596e5a72a5df',
 '0xdfe3fedc5c7679be42c3d393e99d4b55247b73c4',
 '0x59a513a782dd4c1a98ef22932c52191129f5afc8',
 '0x94d0f4a485b7670d244224b33e1659a8b1959e65',
 '0x355e5ae20cc1a3a2164f818e4bac8d22dd72a038',
 '0x1c144e30f

In [21]:
# Per-wallet PnL breakdown
if not fills.empty:
    wallet_pnl_df = (
        fills.groupby("wallet")
        .agg(
            filled_orders=("order_id", "nunique"),
            fill_rows=("order_id", "count"),
            net_pnl_usdc=("fill_pnl_usdc", "sum"),
            total_qty=("fill_qty", "sum"),
            win_rate=("token_winner", lambda s: fills.loc[s.index].groupby("order_id")["token_winner"].first().mean()),
        )
        .assign(win_rate=lambda d: (d["win_rate"] * 100).round(1))
        .sort_values("net_pnl_usdc", ascending=False)
        .reset_index()
    )
    display(wallet_pnl_df)


,wallet,filled_orders,fill_rows,net_pnl_usdc,total_qty,win_rate
0,0xbacd00c9080a82ded56f504ee8810af732b0ab35,3236,7182,122489.571198,461146.736349,64.1
1,0x12d6cccfc7470a3f4bafc53599a4779cbf2cf2a8,3420,6858,81667.750468,478492.638884,50.8
2,0x7c3db723f1d4d8cb9c550095203b686cb11e5c6b,2258,7225,18811.108334,626925.205339,56.3
3,0xc02147dee42356b7a4edbb1c35ac4ffa95f61fa8,747,1457,8979.187088,56664.495063,80.5
4,0x24c8cf69a0e0a17eee21f69d29752bfa32e823e1,860,2004,7868.128097,250906.941785,53.7
...,...,...,...,...,...,...
162,0x0e3226217752d7247c67b870b72b99ba3e20535b,205,370,-2804.933093,12954.819258,46.8
163,0xc7b8a3f1033a9efa9d912b60885c1d2c1e012c1f,598,985,-3148.245446,42918.844811,22.7
164,0xe35dfe51d8288ea409ea18a139f188a83b33f5d5,175,353,-3421.783721,9969.621478,68.6
165,0x4bbe10ba5b7f6df147c0dae17b46c44a6e562cf3,1584,2868,-5315.544243,107878.130331,70.6


In [22]:
wallet_pnl_df.head(10)['wallet'].tolist()

['0xbacd00c9080a82ded56f504ee8810af732b0ab35',
 '0x12d6cccfc7470a3f4bafc53599a4779cbf2cf2a8',
 '0x7c3db723f1d4d8cb9c550095203b686cb11e5c6b',
 '0xc02147dee42356b7a4edbb1c35ac4ffa95f61fa8',
 '0x24c8cf69a0e0a17eee21f69d29752bfa32e823e1',
 '0x22e4248bdb066f65c9f11cd66cdd3719a28eef1c',
 '0x87650b9f63563f7c456d9bbcceee5f9faf06ed81',
 '0x38d812aff0b79f3bf5da2a477f780bcc163eea7c',
 '0xffb0b9b292e406fd250854a35a0c9bd5612afa37',
 '0x43440ab002eaac9fede6f9d21bea96d84228f90d']

In [23]:
# Orders that were NOT filled at all
filled_order_ids = set(fills["order_id"].unique()) if not fills.empty else set()
unfilled_triggers = triggers[~triggers["order_id"].isin(filled_order_ids)]
print(f"Unfilled orders    : {len(unfilled_triggers):,} ({len(unfilled_triggers)/max(len(triggers),1)*100:.1f}% of all triggers)")

# Partially filled orders (filled but qty_filled < qty_ordered)
if not fills.empty:
    filled_trig = triggers[triggers["order_id"].isin(filled_order_ids)].copy()
    partial_mask = filled_trig["qty_filled"] < filled_trig["qty_ordered"] - 1e-9
    partial_fills = filled_trig[partial_mask]
    print(f"Partially filled   : {len(partial_fills):,} ({len(partial_fills)/max(len(filled_trig),1)*100:.1f}% of filled orders)")
    print(f"Fully filled       : {len(filled_trig)-len(partial_fills):,}")

if not unfilled_triggers.empty:
    print("\nUnfilled breakdown by side:")
    display(unfilled_triggers.groupby("side").size().rename("count").reset_index())


Unfilled orders    : 75,637 (50.0% of all triggers)
Partially filled   : 12,082 (16.0% of filled orders)
Fully filled       : 63,644

Unfilled breakdown by side:


,side,count
0,BUY,75637


In [24]:
triggers[triggers['order_id'] == 34]

,row_type,order_id,wallet,condition_id,outcome,side,dt,tx_hash,price,order_price,qty_ordered,qty_filled,fill_qty,fill_source,token_winner,exec_price,fill_pnl_usdc,wallet_trade_pnl,shard
42,trigger,34,0xd6eef76188c7068719ac881460b7431e1ae137a2,0x1597a8e275ae13b3be4a1875362a44056bdd5a0b72ca...,No,BUY,2026-02-16 00:50:53+00:00,0x55b2685815695bf24e7d53ffdfecde272419e64a62fc...,0.076,0.076,2.647427,0.0,NaN,None,False,NaN,NaN,-0.201204,1


In [25]:
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 100)
fills[fills['order_id'] == 34]

,row_type,order_id,wallet,condition_id,outcome,side,dt,tx_hash,price,order_price,qty_ordered,qty_filled,fill_qty,fill_source,token_winner,exec_price,fill_pnl_usdc,wallet_trade_pnl,shard


In [26]:
fills.groupby("wallet")["fill_pnl_usdc"].sum().sort_values(ascending=False).head(100)

wallet
0xbacd00c9080a82ded56f504ee8810af732b0ab35    122489.571198
0x12d6cccfc7470a3f4bafc53599a4779cbf2cf2a8     81667.750468
0x7c3db723f1d4d8cb9c550095203b686cb11e5c6b     18811.108334
0xc02147dee42356b7a4edbb1c35ac4ffa95f61fa8      8979.187088
0x24c8cf69a0e0a17eee21f69d29752bfa32e823e1      7868.128097
0x22e4248bdb066f65c9f11cd66cdd3719a28eef1c      6670.746799
0x87650b9f63563f7c456d9bbcceee5f9faf06ed81      5447.026117
0x38d812aff0b79f3bf5da2a477f780bcc163eea7c      5114.797155
0xffb0b9b292e406fd250854a35a0c9bd5612afa37      4669.270293
0x43440ab002eaac9fede6f9d21bea96d84228f90d      4635.663652
0x6bab41a0dc40d6dd4c1a915b8c01969479fd1292      4416.219332
0xd44e974a3edb232aa4aedbdcc59792b76a5f67e2      4029.767114
0x6ffb4354cbe6e0f9989e3b55564ec5fb8646a834      3802.202447
0x9cb98fc57fe8a3bdf75c3b48bd419e5d262f7b80      3707.901471
0x8597ca63e722d6216bfc3057591fdc67ec49daee      3648.225779
0xfbff8d110ff1a34e486d791b389ea4d5f8cc5830      3632.194561
0x68c24bf4a8ad4d79a6fe4b8eec6f93a

In [27]:
trigger_wallets = triggers[["order_id", "wallet"]].set_index("order_id")["wallet"]

fills_with_wallet = fills.merge(trigger_wallets, on="order_id", how="left", suffixes=("", "_trigger"))

fills_with_wallet['notional'] = fills_with_wallet['fill_qty'] * fills_with_wallet['exec_price']

fills_with_wallet.groupby(["wallet", "condition_id"]).agg(
    pnl=("fill_pnl_usdc", "sum"),
    orders_filled=("order_id", "nunique")
    ).sort_values(['wallet', "pnl"], ascending=[True, False]).head(1)

,,pnl,orders_filled
wallet,condition_id,,
0x000d257d2dc7616feaef4ae0f14600fdf50a758e,0x61ce3773237a948584e422de72265f937034af418a8b703e3a860ea62e59ff36,554.477074,48


In [28]:
fills_with_wallet.head()

,row_type,order_id,wallet,condition_id,outcome,side,dt,tx_hash,price,order_price,...,qty_filled,fill_qty,fill_source,token_winner,exec_price,fill_pnl_usdc,wallet_trade_pnl,shard,wallet_trigger,notional
0,fill,1,0xd6eef76188c7068719ac881460b7431e1ae137a2,0xf86191c4b660e0f0f4669eae5638474277a3bb31e5c34e6118e5eb19e00a11cd,No,BUY,2026-02-16 00:00:51+00:00,0x0b8f56c045cc516e71896d5cf7f99a83674e137e5192bf14e8830f4eea47287c,0.960,NaN,...,NaN,0.004241,same_token,True,0.961,0.000161,NaN,f,0xd6eef76188c7068719ac881460b7431e1ae137a2,0.004076
1,fill,3,0xd6eef76188c7068719ac881460b7431e1ae137a2,0xbebb1c8c6996bdd0783db8891f4cb764afe301c7393ae19cad4c81b0d9e25e11,Yes,BUY,2026-02-16 00:01:41+00:00,0xf2f6a76c3c0c2a1d79d01aaf557e648c61ec3c988d28d0efa9a3922a1a5b5a71,0.910,NaN,...,NaN,2.170000,same_token,True,0.910,0.193325,NaN,b,0xd6eef76188c7068719ac881460b7431e1ae137a2,1.974700
2,fill,4,0xd6eef76188c7068719ac881460b7431e1ae137a2,0xbebb1c8c6996bdd0783db8891f4cb764afe301c7393ae19cad4c81b0d9e25e11,Yes,BUY,2026-02-16 00:02:09+00:00,0xca8ef7824274e6e5e97d02f83f95d06d1985eacac3ba1c908032330c2bb35e83,0.910,NaN,...,NaN,2.140000,same_token,True,0.910,0.190653,NaN,b,0xd6eef76188c7068719ac881460b7431e1ae137a2,1.947400
3,fill,14,0xd6eef76188c7068719ac881460b7431e1ae137a2,0xf7ddbcf3f56543a8fa36cad800d0390d1e423f0cc049cdb5a14592eabe87f3c2,Yes,BUY,2026-02-16 00:26:01+00:00,0x6b97961eaec76ec7d20cdb639dda75076148dbd282f1c4cd535e6e2a641366d6,0.040,NaN,...,NaN,24.160000,opposite_token,False,0.050,-1.209208,NaN,f,0xd6eef76188c7068719ac881460b7431e1ae137a2,1.208000
4,fill,17,0xbacd00c9080a82ded56f504ee8810af732b0ab35,0x9d52a1ad771507479cb17d8748705f5d0556796669fd7dec42e6766400a8f445,Yes,BUY,2026-02-16 00:29:01+00:00,0x13c69501d8e82b2a80d535058ed0ca2a0bcb8ce34af6fefda1b7b2504d2358fe,0.544,NaN,...,NaN,100.000000,same_token,False,0.565,-56.556500,NaN,9,0xbacd00c9080a82ded56f504ee8810af732b0ab35,56.500000


In [29]:
wallet_pnls = (
    fills_with_wallet
    .groupby(["wallet", "condition_id"])
    .agg(
        pnl=("fill_pnl_usdc", "sum"),
        orders_filled=("order_id", "nunique"),
        notional=("notional", "sum"),
    )
    .groupby("wallet")
    .agg(
        pnl=("pnl", "sum"),
        notional=("notional", "sum"),
        unique_conditions=("pnl", "size"),  # ✅ count rows
        total_orders_filled=("orders_filled", "sum"),
    )
    .sort_values("pnl", ascending=False)
)

In [30]:
wallet_pnls.head()

,pnl,notional,unique_conditions,total_orders_filled
wallet,,,,
0xbacd00c9080a82ded56f504ee8810af732b0ab35,122489.571198,127588.069793,126,3236
0x12d6cccfc7470a3f4bafc53599a4779cbf2cf2a8,81667.750468,111907.761671,48,3420
0x7c3db723f1d4d8cb9c550095203b686cb11e5c6b,18811.108334,87559.116865,158,2258
0xc02147dee42356b7a4edbb1c35ac4ffa95f61fa8,8979.187088,31958.163370,56,747
0x24c8cf69a0e0a17eee21f69d29752bfa32e823e1,7868.128097,37296.212157,34,860


In [31]:
MAX_WALLET_EXPOSURE = 1000.0  # USDC
wallet_pnls['pnl_limited'] = np.where(
    wallet_pnls['notional'] <= MAX_WALLET_EXPOSURE,
    wallet_pnls['pnl'],
    wallet_pnls['pnl'] * MAX_WALLET_EXPOSURE / wallet_pnls['notional']
)

In [32]:
wallet_pnls

,pnl,notional,unique_conditions,total_orders_filled,pnl_limited
wallet,,,,,
0xbacd00c9080a82ded56f504ee8810af732b0ab35,122489.571198,127588.069793,126,3236,960.039378
0x12d6cccfc7470a3f4bafc53599a4779cbf2cf2a8,81667.750468,111907.761671,48,3420,729.777356
0x7c3db723f1d4d8cb9c550095203b686cb11e5c6b,18811.108334,87559.116865,158,2258,214.838945
0xc02147dee42356b7a4edbb1c35ac4ffa95f61fa8,8979.187088,31958.163370,56,747,280.966931
0x24c8cf69a0e0a17eee21f69d29752bfa32e823e1,7868.128097,37296.212157,34,860,210.963196
...,...,...,...,...,...
0x0e3226217752d7247c67b870b72b99ba3e20535b,-2804.933093,7325.814772,36,205,-382.883431
0xc7b8a3f1033a9efa9d912b60885c1d2c1e012c1f,-3148.245446,10828.687899,102,598,-290.731941
0xe35dfe51d8288ea409ea18a139f188a83b33f5d5,-3421.783721,9012.285505,58,175,-379.679907


In [33]:
result = (
    fills_with_wallet
    # 1️⃣ PnL per market (condition) per wallet
    .groupby(["wallet", "condition_id"])["fill_pnl_usdc"]
    .sum()
    
    # 2️⃣ Then per wallet
    .groupby("wallet")
    .agg(
        unique_conditions="count",   # number of markets traded
        median_market_pnl="median",  # median PnL across markets
    )
    .sort_values("unique_conditions", ascending=False)
    .head(20)
)

In [34]:
result.sort_values("median_market_pnl", ascending=False).head(100)

,unique_conditions,median_market_pnl
wallet,,
0xc7b8a3f1033a9efa9d912b60885c1d2c1e012c1f,102,3.735544
0x7c3db723f1d4d8cb9c550095203b686cb11e5c6b,158,1.909404
0x9cdda449d7cedf1072d74982a5dc2349df3d3e97,147,1.471500
0x6ffb4354cbe6e0f9989e3b55564ec5fb8646a834,154,0.905551
0x41cb8653fd75ebf2a46741d224dc596e5a72a5df,146,0.792848
0xb03b826a4fc9893b35d3ddf4f11be824525b6ca1,97,0.729334
0x9e8ecc4cb3c4e48f544cba2fbbb252a6a65e8db8,267,0.514256
0xbacd00c9080a82ded56f504ee8810af732b0ab35,126,0.163393
0x134a63b764ac7b008356e8db1857db94e6b09e42,103,0.139459


In [35]:
fills_with_wallet.head()

,row_type,order_id,wallet,condition_id,outcome,side,dt,tx_hash,price,order_price,...,qty_filled,fill_qty,fill_source,token_winner,exec_price,fill_pnl_usdc,wallet_trade_pnl,shard,wallet_trigger,notional
0,fill,1,0xd6eef76188c7068719ac881460b7431e1ae137a2,0xf86191c4b660e0f0f4669eae5638474277a3bb31e5c34e6118e5eb19e00a11cd,No,BUY,2026-02-16 00:00:51+00:00,0x0b8f56c045cc516e71896d5cf7f99a83674e137e5192bf14e8830f4eea47287c,0.960,NaN,...,NaN,0.004241,same_token,True,0.961,0.000161,NaN,f,0xd6eef76188c7068719ac881460b7431e1ae137a2,0.004076
1,fill,3,0xd6eef76188c7068719ac881460b7431e1ae137a2,0xbebb1c8c6996bdd0783db8891f4cb764afe301c7393ae19cad4c81b0d9e25e11,Yes,BUY,2026-02-16 00:01:41+00:00,0xf2f6a76c3c0c2a1d79d01aaf557e648c61ec3c988d28d0efa9a3922a1a5b5a71,0.910,NaN,...,NaN,2.170000,same_token,True,0.910,0.193325,NaN,b,0xd6eef76188c7068719ac881460b7431e1ae137a2,1.974700
2,fill,4,0xd6eef76188c7068719ac881460b7431e1ae137a2,0xbebb1c8c6996bdd0783db8891f4cb764afe301c7393ae19cad4c81b0d9e25e11,Yes,BUY,2026-02-16 00:02:09+00:00,0xca8ef7824274e6e5e97d02f83f95d06d1985eacac3ba1c908032330c2bb35e83,0.910,NaN,...,NaN,2.140000,same_token,True,0.910,0.190653,NaN,b,0xd6eef76188c7068719ac881460b7431e1ae137a2,1.947400
3,fill,14,0xd6eef76188c7068719ac881460b7431e1ae137a2,0xf7ddbcf3f56543a8fa36cad800d0390d1e423f0cc049cdb5a14592eabe87f3c2,Yes,BUY,2026-02-16 00:26:01+00:00,0x6b97961eaec76ec7d20cdb639dda75076148dbd282f1c4cd535e6e2a641366d6,0.040,NaN,...,NaN,24.160000,opposite_token,False,0.050,-1.209208,NaN,f,0xd6eef76188c7068719ac881460b7431e1ae137a2,1.208000
4,fill,17,0xbacd00c9080a82ded56f504ee8810af732b0ab35,0x9d52a1ad771507479cb17d8748705f5d0556796669fd7dec42e6766400a8f445,Yes,BUY,2026-02-16 00:29:01+00:00,0x13c69501d8e82b2a80d535058ed0ca2a0bcb8ce34af6fefda1b7b2504d2358fe,0.544,NaN,...,NaN,100.000000,same_token,False,0.565,-56.556500,NaN,9,0xbacd00c9080a82ded56f504ee8810af732b0ab35,56.500000
